# Image Preprocessing and Augmentation

## Overview

Images require specialised preprocessing before deep learning: normalisation, resizing, and augmentation. Augmentation artificially expands training data by applying label-preserving transformations — critical for small ecological image datasets.

**Preprocessing pipeline:**
```
Raw image (H×W×C, uint8 0-255)
  → Resize to fixed dimensions
  → Convert to float32
  → Normalise to [0,1] or standardise per channel
  → (Training only) Augment
  → torch Tensor (C×H×W)
```

**Augmentation principles:**
- Only apply transformations that preserve the label
- Augment training data only — never validation or test
- Stronger augmentation helps more when data is scarce
- Domain knowledge guides valid transforms (e.g. vertical flip invalid for landscape photos)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
import io

rng = np.random.default_rng(42)

def make_fake_image(label, size=128, seed=0):
    rng_l = np.random.default_rng(seed)
    # Simulate: satellite-like patches — healthy=green, degraded=brown
    img = np.zeros((size, size, 3), dtype=np.uint8)
    if label == 0:   # healthy riparian: green with variation
        img[:,:,0] = rng_l.integers(40, 80,  (size,size))   # R
        img[:,:,1] = rng_l.integers(100,160, (size,size))   # G
        img[:,:,2] = rng_l.integers(30, 70,  (size,size))   # B
    else:            # degraded: brown/bare
        img[:,:,0] = rng_l.integers(120,180, (size,size))   # R
        img[:,:,1] = rng_l.integers(80, 120, (size,size))   # G
        img[:,:,2] = rng_l.integers(30, 60,  (size,size))   # B
    # Add spatial texture
    noise = rng_l.integers(-20, 20, (size, size, 3))
    img   = np.clip(img.astype(int) + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(img)

# Create small sample of fake images
imgs   = [make_fake_image(lbl, seed=i) for i, lbl in enumerate([0,0,1,1,0,1])]
labels = [0,0,1,1,0,1]
print(f"Images: {len(imgs)}, size={imgs[0].size}, mode={imgs[0].mode}")
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for ax, img, lbl in zip(axes.ravel(), imgs, labels):
    ax.imshow(img); ax.set_title(['Healthy','Degraded'][lbl]); ax.axis('off')
plt.suptitle('Simulated Riparian Patch Images'); plt.tight_layout(); plt.show()

---
## Standard Transforms

In [ ]:
# ImageNet normalisation constants
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training transform: resize + augment + normalise
train_transform = T.Compose([
    T.Resize((64, 64)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomRotation(degrees=15),
    T.ToTensor(),                         # uint8 [0,255] -> float [0,1], HWC->CHW
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
# Validation transform: resize + normalise only (no augmentation)
val_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
# Demonstrate
sample_img = imgs[0]
fig, axes  = plt.subplots(2, 5, figsize=(15, 6))
axes[0,0].imshow(sample_img); axes[0,0].set_title('Original'); axes[0,0].axis('off')
for i in range(1, 5):
    aug = T.Compose([
        T.Resize((64,64)), T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        T.RandomRotation(15),
    ])(sample_img)
    axes[0,i].imshow(aug); axes[0,i].set_title(f'Aug {i}'); axes[0,i].axis('off')
tensor = train_transform(sample_img)
axes[1,0].imshow(tensor.permute(1,2,0).numpy().clip(0,1))
axes[1,0].set_title('As tensor (denorm)'); axes[1,0].axis('off')
for ax in axes[1,1:]: ax.axis('off')
plt.suptitle('Augmentation Examples (same image, different transforms)')
plt.tight_layout(); plt.show()

---
## Custom Dataset Class

In [ ]:
class RiparianDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img   = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

# Build datasets with different transforms for train/val
all_imgs   = [make_fake_image(l, seed=i) for i, l in enumerate([0]*20 + [1]*20)]
all_labels = [0]*20 + [1]*20
split      = 32
train_ds = RiparianDataset(all_imgs[:split],  all_labels[:split],  train_transform)
val_ds   = RiparianDataset(all_imgs[split:],  all_labels[split:],  val_transform)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)
Xb, yb = next(iter(train_dl))
print(f"Batch: X={Xb.shape}, y={yb.shape}")
print(f"X dtype: {Xb.dtype}, range: [{Xb.min():.2f}, {Xb.max():.2f}]")
print("\nKey: train_ds uses train_transform; val_ds uses val_transform")
print("Same underlying images -- only transforms differ")

---
## Augmentation Visualisation and Impact

In [ ]:
# Show effect of each augmentation type individually
augmentations = {
    'Original':       T.Compose([T.Resize((64,64))]),
    'HFlip':          T.Compose([T.Resize((64,64)), T.RandomHorizontalFlip(p=1.0)]),
    'VFlip':          T.Compose([T.Resize((64,64)), T.RandomVerticalFlip(p=1.0)]),
    'ColorJitter':    T.Compose([T.Resize((64,64)), T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)]),
    'Rotation 30°':   T.Compose([T.Resize((64,64)), T.RandomRotation(30)]),
    'GaussianBlur':   T.Compose([T.Resize((64,64)), T.GaussianBlur(kernel_size=5)]),
}
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, tfm) in zip(axes.ravel(), augmentations.items()):
    aug_img = tfm(imgs[0])
    ax.imshow(aug_img); ax.set_title(name); ax.axis('off')
plt.suptitle('Individual Augmentation Types'); plt.tight_layout(); plt.show()
print("Augmentation selection guidelines for ecological imagery:")
print("  HFlip/VFlip:   valid for satellite/aerial patches")
print("  ColorJitter:   valid; simulates lighting/sensor variation")
print("  Rotation:      valid for top-down views; careful for ground photos")
print("  GaussianBlur:  simulates focus variation; generally safe")
print("  Cutout/Mixup:  advanced; useful for small datasets")

In [ ]:
# Normalisation: demonstrate effect on training stability
import torch.nn as nn, torch.optim as optim

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(2),
            nn.Flatten(), nn.Linear(32*4, 2))
    def forward(self, x): return self.net(x)

# Compare: unnormalised vs normalised inputs
results_norm = {}
for use_norm in [False, True]:
    tfm = T.Compose([T.Resize((32,32)), T.ToTensor(),
                     *([T.Normalize(IMAGENET_MEAN, IMAGENET_STD)] if use_norm else [])])
    ds  = RiparianDataset(all_imgs[:split], all_labels[:split], tfm)
    dl  = DataLoader(ds, batch_size=8, shuffle=True, num_workers=0)
    model = TinyCNN()
    opt   = optim.SGD(model.parameters(), lr=0.01)
    crit  = nn.CrossEntropyLoss()
    losses = []
    for ep in range(20):
        ep_loss = []
        for Xb, yb in dl:
            opt.zero_grad(); loss = crit(model(Xb), yb)
            loss.backward(); opt.step(); ep_loss.append(loss.item())
        losses.append(np.mean(ep_loss))
    results_norm['Normalised' if use_norm else 'Unnormalised'] = losses
fig, ax = plt.subplots(figsize=(9,4))
for label, losses in results_norm.items():
    ax.plot(losses, label=label, lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Normalisation Effect on Training Stability'); ax.legend()
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Applying augmentation to the validation and test sets**  
Augmentation is a training-time technique to increase effective dataset size. Applying random flips, crops, or colour jitter to validation images produces different results on each evaluation call, making metrics non-reproducible. Always use a deterministic transform (resize + normalise only) for validation and test.

**2. Choosing augmentations that change the label**  
For satellite land-cover classification, a vertical flip of a "water body" image may now resemble a different class depending on orientation. For digit recognition, rotating a "6" produces a "9". Always sanity-check that each augmentation preserves the semantic label for your specific task.

**3. Normalising with incorrect or inconsistent statistics**  
If you normalise training images with their own per-batch statistics but normalise test images with ImageNet statistics (or vice versa), train and test distributions differ. Decide on one normalisation strategy and apply it consistently everywhere — including at inference time in production.

**4. Not converting PIL Images to float before normalising manually**  
`T.ToTensor()` converts uint8 [0,255] to float32 [0,1] AND transposes from HWC to CHW format. Skipping `ToTensor()` and calling `Normalize` on an integer tensor produces incorrect results silently. Always apply `ToTensor()` before `Normalize()`.

**5. Using `num_workers > 0` on Windows without `if __name__ == "__main__"` guard**  
PyTorch DataLoader spawns worker processes for parallel data loading. On Windows, this requires the script to be protected by `if __name__ == "__main__":`. Without this guard, each worker re-imports the script and spawns more workers, causing infinite recursion. Use `num_workers=0` in notebooks or add the guard in scripts.
---
*python_methods_library - Samantha McGarrigle*